### 08 LCEL 체인에 메모리 추가하기

- 임의의 체인에 메모리를 추가하는 방법! 현재 메모리 클래스를 사용할 수 있지만 수동으로 연결해야 한다

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

- **MessagePLaceholder**
  - 프롬프트 템플릿에서 대화 기록을 포함할 위치를 지정하는 데 사용
  - 빈 공간을 잡아놨다가 대화를 이어 나가면 대화 기록이 쌓인다
- `variable_name` 매개변수는 대화 기록이 프롬프트에 삽입될 때 참조되는 변수 이름을 정의

In [2]:
from operator import itemgetter
from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI


# ChatOpenAI 모델을 초기화합니다.
model = ChatOpenAI()

# 대화형 프롬프트를 생성합니다. 이 프롬프트는 시스템 메시지, 이전 대화 내역, 그리고 사용자 입력을 포함합니다.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful chatbot"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

- 대화내용을 저장할 메모리인 `ConversationBufferMemory` 생성하고 `return_messages` 매개변수를 `True`로 설정하여, 생성된 인스턴스가 메시지를 리스트 형식으로 반환
- `memory_key` 설정: 추후 Chain 의 `prompt` 안에 대입될 key. 변경하여 사용 가능

In [3]:
memory = ConversationBufferMemory(return_messages=True, memory_key='chat_history')

/var/folders/5h/jssvspks5fn6v_wtpcsmpp580000gn/T/ipykernel_99486/2335556379.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(return_messages=True, memory_key='chat_history')


- 저장된 대화기록을 확인. 아직 저장하지 않았으므로, 대화기록은 비어있다

In [4]:
memory.load_memory_variables({})

{'chat_history': []}

- 메모리에 저장된 대화기록을 추출하고, RunnablePassThrough를 통해 다른 프로세스에 전달해 이를 토대로 답변할 수 있게 만들어야한다
- `RunnablePassthrough.assign`을 사용하여 `chat_history` 변수에 `memory.load_memory_variables` 함수의 결과를 할당하고, 이 결과에서 `chat_history` 키에 해당하는 값을 추출

In [ ]:
# RunnablePassthrough.assign은 입력 데이터를 그대로 통과시키면서 새로운 키-값 쌍을 추가하는 역할
runnable = RunnablePassthrough.assign(
    chat_history=RunnableLambda(memory.load_memory_variables)
    | itemgetter("chat_history")  # memory_key 와 동일하게 입력합니다.
)

- **RunnableLambda**는 함수를 실행 가능한 객체로 매핑시켜주는 역할
- `memory.load_memory_variables`는 현재 메모리에 저장된 대화 기록을 반환
- 지금까지 쌓아왔던 대화기록을 불러오는 함수를 호출해서 chat_history라는 키에 대화 기록 대입

In [ ]:
chat_history = RunnableLambda(memory.load_memory_variables)

- itemgetter는 일단 무시하고 runnale 객체부터 본다
- `RunnablePassthrough.assign`은 입력 데이터는 그대로 유지하고, chat_history라는 키와 계산된 값의 쌍을 추가. 두 값이 runnable 객체에 할당
- 출력 결과에서 input은 나중에 사용자의 질문이 되고, chat_history는 지금까지 해온 대화 기록

In [5]:
runnable = RunnablePassthrough.assign(
    chat_history=RunnableLambda(memory.load_memory_variables)
#    | itemgetter("chat_history")
)
runnable.invoke({'input': 'hi'})

{'input': 'hi', 'chat_history': {'chat_history': []}}

- 최종 답변에서 chat_history는 리스트 형식으로 나와야 한다
- 현재는 dictionary 형태니깐 itemgetter로 키에 해당하는 값만 가져온다

In [6]:
runnable = RunnablePassthrough.assign(
    chat_history=RunnableLambda(memory.load_memory_variables)
    | itemgetter("chat_history")
)
runnable.invoke({'input': 'hi'})

{'input': 'hi', 'chat_history': []}

In [7]:
chain = runnable | prompt | model

- runnable의 input 값은 prompt에서 human 메시지로, chat_history는 MessagesPlaceholder로 전달
- 그런 다음 prompt에 채워진 값이 model에 주입

In [8]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful chatbot"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

In [9]:
response = chain.invoke({"input": "만나서 반갑습니다. 제 이름은 테디입니다."})
print(response.content)  # 생성된 응답을 출력합니다.

만나서 반갑습니다, 테디님! 어떻게 도와드릴까요?


- 여전히 chat_history는 빈 리스트

In [11]:
memory.load_memory_variables({})

{'chat_history': []}

- 메모리에 매번 저장되려면 `save_context` 함수를 호출하여 입력 데이터(`inputs`)와 응답 내용(`response.content`)으로 이루어진 모든 대화를 명시적으로 저장

In [12]:
# 입력된 데이터와 응답 내용을 메모리에 저장합니다.
memory.save_context(
    {"human": "만나서 반갑습니다. 제 이름은 테디입니다."}, {"ai": response.content}
)

# 저장된 대화기록을 출력합니다.
memory.load_memory_variables({})

{'chat_history': [HumanMessage(content='만나서 반갑습니다. 제 이름은 테디입니다.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='만나서 반갑습니다, 테디님! 어떻게 도와드릴까요?', additional_kwargs={}, response_metadata={})]}

In [13]:
# 이름을 기억하고 있는지 추가 질의합니다.
response = chain.invoke({"input": "제 이름이 무엇이었는지 기억하세요?"})
# 답변을 출력합니다.
print(response.content)

네, 테디님의 이름을 기억하고 있습니다. 무엇을 도와드릴까요?


- 여기서 살펴본 방식으로 여러 대화 내용을 기억하는 멀티턴 방식을 구현하려면 매번 저장해야해서 사용하기 불편
- 이후에 메모리를 더 직관적으로 효율적으로 관리 가능한 `RunnableWithMessageHistory` 학습